# ДЗ 7. Срезы orders.csv: groupby, pivot_table, merge
Срок: **до воскресенья 29.11, 24:00**.
Сдача: папка `hw07` репозитория `dsa-homework` (`hw07.ipynb`): `git add hw07` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать. Рядом с каждой долей или средним указывайте, на скольких заказах оно посчитано.

Инструменты: всё с уроков 1–7. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
t = pd.DataFrame({"c": ["a", "b", "a"], "x": [1, 2, 3]})
print(t.groupby("c")["x"].sum().tolist())
# 2
t = pd.DataFrame({"c": ["a", "a", "b"], "d": [1, None, 3]})
print(t.groupby("c")["d"].count().tolist(), t.groupby("c").size().tolist())
# 3
t = pd.DataFrame({"c": ["a", "a", "b"], "m": [9, 9, 10], "x": [10, 30, 5]})
print(t.pivot_table(index="c", columns="m", values="x").loc["a", 9])
# 4
a = pd.DataFrame({"id": [1, 2, 3]})
b = pd.DataFrame({"id": [1, 2], "ch": ["x", "y"]})
print(len(a.merge(b, on="id")), len(a.merge(b, on="id", how="left")))
# 5
b = pd.DataFrame({"id": [1, 1, 2], "ch": ["x", "x", "y"]})
print(len(a.merge(b, on="id", how="left")))
```

In [4]:
import pandas as pd
t = pd.DataFrame({"c": ["a", "b", "a"], "x": [1, 2, 3]})
print(t.groupby("c")["x"].sum().tolist())

[4, 2]


In [5]:
t = pd.DataFrame({"c": ["a", "a", "b"], "d": [1, None, 3]})
print(t.groupby("c")["d"].count().tolist(), t.groupby("c").size().tolist())

[1, 1] [2, 1]


In [6]:
t = pd.DataFrame({"c": ["a", "a", "b"], "m": [9, 9, 10], "x": [10, 30, 5]})
print(t.pivot_table(index="c", columns="m", values="x").loc["a", 9])

20.0


In [7]:
a = pd.DataFrame({"id": [1, 2, 3]})
b = pd.DataFrame({"id": [1, 2], "ch": ["x", "y"]})
print(len(a.merge(b, on="id")), len(a.merge(b, on="id", how="left")))

2 3


In [8]:
b = pd.DataFrame({"id": [1, 1, 2], "ch": ["x", "x", "y"]})
print(len(a.merge(b, on="id", how="left")))

4


Ваши ответы и объяснения:

1. [4, 2] | groupby сортирует группы по алфавиту ("a", "b") и суммирует значения "x": 1+3=4 и 2
2. [1, 1] [2, 1] | count пропускает None, а size считает все строки
3. 20.0 | pivot_table не указан параметр aggfunc, поэтому по умолчанию считается среднее арифметическое двух строк для a и 9: (10 + 30) / 2 = 20.0
4. 2 3 | по умолчанию merge работает в режиме how="inner" и оставляет только общие id (1 и 2, поэтому 2 строки), а how="left" сохраняет все строки левой таблицы a (1, 2, 3, подставляя NaN для 3, поэтому 3 строки)
5. 4 | из-за дубликата id = 1 в справочнике b строка id = 1 раздваивается (2 строки), плюс id = 2 (1 строка) и id = 3 с NaN (1 строка)

In [3]:
# Загрузка orders.csv и customers.csv: в Colab нажмите «Выбрать файлы» и укажите оба файла из LMS.
# Если файлы уже лежат рядом с ноутбуком, загрузка пропускается.
import os
if not (os.path.exists("orders.csv") and os.path.exists("customers.csv")):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv и customers.csv в одну папку с ноутбуком")
print("orders.csv:", os.path.exists("orders.csv"), "| customers.csv:", os.path.exists("customers.csv"))

orders.csv: True | customers.csv: True


In [4]:
# Очистка урока 6 одной ячейкой: суммы, города, дубли, даты, TEST
import pandas as pd
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

df = pd.read_csv("orders.csv")
df["amount"] = pd.to_numeric(df["amount"].str.replace(" ", ""))
df["city"] = df["city"].str.strip().str.lower().map(CITY)
df = df.drop_duplicates()
df["order_date"] = pd.to_datetime(df["order_date"])
df["month"] = df["order_date"].dt.month
df = df[df["customer_id"] != "TEST"].copy()    # copy: дальше добавим столбцы
df.shape

(5280, 9)

In [5]:
df["cancel"] = df["status"] == "отменён"
paid = df[df["status"] == "оплачен"]
cu = pd.read_csv("customers.csv")
len(paid), cu.shape

(4739, (2200, 4))

Можно вместо ячейки очистки взять свою функцию `clean()` из ДЗ 6. Проверка: 5 280 строк, 4 739 оплаченных.

## [База] 1. Сводка по городам одной таблицей
Через `agg` с именами: число заказов, доля отмен, средний срок доставки по городам. Добавьте столбец «выручка оплаченных»: `by_city["revenue"] = paid.groupby("city")["amount"].sum()` (Pandas выровняет по ярлыкам, урок 5). Отсортируйте по выручке.
Какой город выделяется сразу по двум показателям?

In [10]:
by_city = df.groupby("city").agg(
    orders=("order_id", "count"),
    cancel_share=("cancel", "mean"),
    avg_days=("delivery_days", "mean")
)

In [11]:
# 1
by_city["revenue"] = paid.groupby("city")["amount"].sum()

In [13]:
by_city = by_city.sort_values("revenue", ascending=False).round(3)
by_city

,orders,cancel_share,avg_days,revenue
city,,,,
Алматы,2116,0.078,1.997,29144090.0
Астана,1339,0.088,2.019,17884750.0
Шымкент,776,0.206,4.451,8602660.0
Караганда,519,0.091,1.941,7020200.0
Актобе,387,0.106,2.070,5117380.0
Атырау,143,0.077,2.036,1893850.0


Шымкент (на 776 заказах) резко выделяется сразу по двум плохим показателям — доля отмен 20,6% (против 7,7–10,6% в остальных городах) и средний срок доставки 4,45 дня (против ~2 дней в остальных городах). Необходимо проверить работу доставки в Шымкенте.

## [База] 2. ДЗ 4 одной строкой
В ДЗ 4 вы считали категории циклами и NumPy. Повторите через `pivot_table`: выручка оплаченных по категориям и месяцам, изменение в %, средний чек по категориям и месяцам.
Сверьте с ответами ДЗ 4: лидер Электроника 34,4 млн ₸, Одежда −7,2%, средний чек Электроники в октябре 33 600 ₸. Сколько строк кода было и сколько стало?

In [19]:
# 2
cat_rev = paid.pivot_table(index="category", columns="month", values="amount", aggfunc="sum")
cat_rev["change_%"] = (cat_rev[10] / cat_rev[9] - 1) * 100 
print(cat_rev.round(1))

month                 9          10  change_%
category                                     
Дом           4997730.0   5125790.0       2.6
Книги         1065300.0   1140910.0       7.1
Красота       2610950.0   2678990.0       2.6
Одежда        9167100.0   8505730.0      -7.2
Электроника  17402330.0  16968100.0      -2.5


In [20]:
cat_avg = paid.pivot_table(index="category", columns="month", values="amount", aggfunc="mean")
print(cat_avg.round())

month             9        10
category                     
Дом          10499.0  10768.0
Книги         4020.0   3816.0
Красота       6187.0   6159.0
Одежда       13521.0  13046.0
Электроника  32773.0  33600.0


Сверка сошлась: лидер Электроника 34,4 млн ₸, Одежда −7,2%, средний чек Электроники в октябре ~33 600 ₸; вместо ~15–20 строк циклов и NumPy из ДЗ 4 теперь всё решается парой строк pivot_table

## [База] 3. Отмены и срок доставки
По `delivery_days`: число заказов и доля отмен. Во сколько раз отменяют чаще при доставке за 8 дней, чем за 1 день?
Что это значит для Шымкента, где в октябре доставка 5–8 дней?

In [25]:
# 3
by_days = df.groupby("delivery_days").agg(
    orders=("order_id", "count"),
    cancel_share=("cancel", "mean")
)
print(by_days.round(3))

               orders  cancel_share
delivery_days                      
1.0              1557         0.042
2.0              1570         0.080
3.0              1590         0.134
5.0                86         0.221
6.0               120         0.217
7.0                95         0.347
8.0               104         0.433


In [27]:
print((by_days.loc[8, "cancel_share"] / by_days.loc[1, "cancel_share"]).round(3))

10.208


Чем дольше доставка, тем выше доля отмен.Так как в октябре доставка в Шымкенте растянулась до 5–8 дней, именно сбой доставки напрямую вызвал массовые отмены заказов

## [Стандарт] 4. Исправьте код от AI
AI посчитал изменение выручки по городам и пишет менеджеру: «В Шымкенте выручка выросла на 2,4%, проблем нет». В коде **две** ошибки. Найдите их, исправьте и напишите менеджеру правильную цифру.
```python
p = df.pivot_table(index="city", columns="month", values="amount")
p["chg"] = p[10] / p[9] - 1
print(p["chg"].round(3))
```

In [28]:
# 4
p = paid.pivot_table(index="city", columns="month", values="amount", aggfunc="sum")
p["chg"] = p[10] / p[9] - 1
print(p["chg"].round(3))

city
Актобе      -0.014
Алматы       0.010
Астана      -0.020
Атырау       0.050
Караганда   -0.076
Шымкент     -0.113
Name: chg, dtype: float64


1 ошибка - нужно было смотреть не по общей таблице(df.pivot_table) куда попадали и отменённые заказы, а где только отсортированны оплаченные заказы(paid.pivot_table). 2 ошибка - забыл AI поставить функцию aggfunc , по умолчанию она считает среднее, а нам нужна сумма(aggfunc="sum"). 
Реальная выручка в Шымкенте не выросла на 2,4%, а упала на 11,3%

## [Стандарт] 5. Возраст клиентов
Соедините заказы с клиентами (`merge`, `how="left"`, проверьте `len`). Возрастная группа: `m["age_group"] = m["age"] // 10 * 10`.
По группам: число заказов, доля отмен, средний и медианный чек оплаченных.
AI предлагает: «Группа 60 покупает дороже всех и чаще отменяет, сделаем для неё отдельную акцию». Какие возрасты попали в группу 60 и сколько в ней заказов? Подтверждает ли медиана вывод AI?

In [29]:
# 5
m = df.merge(cu, on="customer_id", how="left")
print(len(df), len(m))

5280 5280


In [30]:
m["age_group"] = m["age"] // 10 * 10

In [32]:
paid_m = m[m["status"] == "оплачен"]
by_age = m.groupby("age_group").agg(
    orders=("order_id", "count"), 
    cancel_share=("cancel", "mean"),
)
by_age["avg_check"] = paid_m.groupby("age_group")["amount"].mean()
by_age["med_check"] = paid_m.groupby("age_group")["amount"].median()
print(by_age.round(3))

           orders  cancel_share  avg_check  med_check
age_group                                            
10            229         0.087  14044.737    10700.0
20           1226         0.096  14130.415    10100.0
30           1241         0.111  15142.339    10670.0
40           1170         0.111  14792.971    10050.0
50           1294         0.092  14671.796    10240.0
60            120         0.133  16779.135    10495.0


In [34]:
m[m["age_group"] == 60]["age"].unique()

array([60])

В группу заказов 60 попал только возраст 60 лет, в ней 120 заказов.Подтверждает ли медиана вывод AI: нет! Средний чек там действительно высокий (16 779 ₸),но медианный чек (10 495 ₸) — совершенно обычный (на уровне остальных групп: 10 050 – 10 740 ₸).

## [Стандарт] 6. Повторные покупки
Сколько заказов у каждого клиента (`groupby("customer_id").size()`)? Сколько клиентов сделали 1, 2, 3… заказа (`value_counts().sort_index()`)? Какая доля клиентов купила больше одного раза?
Сколько клиентов заказывали и в сентябре, и в октябре (`groupby("customer_id")["month"].nunique()`)? Сколько клиентов из customers.csv не заказали ничего?

In [9]:
# 6
cust = paid.groupby("customer_id").size()
cust

customer_id
C00001    4
C00002    1
C00003    2
C00004    2
C00005    3
         ..
C02196    1
C02197    3
C02198    1
C02199    2
C02200    1
Length: 1945, dtype: int64

In [11]:
cust.value_counts().sort_index()

1     572
2     568
3     424
4     229
5      96
6      39
7       9
8       7
10      1
Name: count, dtype: int64

In [15]:
print(((cust>1).mean()).round(3))

0.706


In [26]:
order_sep_oct = (paid.groupby("customer_id")["month"].nunique() == 2).sum()
print(order_sep_oct)

946


In [20]:
len(cu["customer_id"])

2200

In [22]:
df["customer_id"].nunique()

2002

In [25]:
print(len(cu["customer_id"]) - df["customer_id"].nunique())

198


Всего оплаченные заказы делали 1 945 клиентов. 1 заказ сделали 572 клиента, 2 заказа — 568, 3 — 424 и далее до 10 заказов. Доля клиентов с повторными покупками (> 1 заказа) — 70,6%. В оба месяца (и в сентябре, и в октябре) покупали 946 клиентов. 198 клиентов из customers.csv не заказали ничего. 

## [Челлендж] 7. Сколько стоил сбой доставки в Шымкенте
Допустим, в октябре в Шымкенте было бы столько же заказов и тот же средний чек оплаченных, а доля отмен осталась как в сентябре. Какой была бы выручка Шымкента в октябре? Сколько компания потеряла из-за отмен?
Сравните потерю с падением всей компании (823 890 ₸). Что было бы с выручкой компании без сбоя?

In [ ]:
# 7

## [Челлендж] 8. Органика −6,0%: сигнал или шум?
Разбейте клиентов из customers.csv **случайно** на три группы в тех же пропорциях, что и каналы (`np.random.default_rng(7)`, `rng.choice`, `map`). Посчитайте изменение выручки оплаченных окт/сент для каждой группы. Повторите 100 раз в цикле.
В каком диапазоне меняется выручка случайной группы размером с органику (5% и 95% перцентили)? Можно ли по «органика −6,0%» делать вывод о SEO?

In [ ]:
# 8